# LiCSBAS Jakarta v9 — consistent wrapped-phase regeneration

This self-contained Colab notebook starts a new processing branch after the historical LiCSAR unwrapped stack failed the fixed Jakarta-ROI factor-5 gate. It does **not** reuse any source unwrapped-phase values. Instead, it restores the verified archive for coherence and metadata, downloads the same unfiltered wrapped-phase product for all 609 pairs from the CEDA archive, and uses LiCSBAS's official `LiCSBAS02to05_unwrap.py` route with pinned `licsar_extra` and SNAPHU to regenerate the entire stack consistently.

The controlled method is: fixed Jakarta ROI `[106.45, -6.54, 107.22, -6.01]`, factor 5, unfiltered wrapped phase for every pair, Goldstein filtering, the recommended single cascade, consistency threshold 0.3, step-12 `--multi_prime`, and the unchanged 1.5-radian loop threshold. GACOS is deliberately not mixed into this first regeneration because the preflight found it for only 78 of 120 acquisitions; using it would silently shorten the network.

The historical 25% rejection ceiling is recorded for comparison but is not treated as a universal LiCSBAS law. The original [LiCSBAS paper](https://doi.org/10.3390/rs12030424) says pair removal is acceptable when sufficient interferograms remain; it defines the 1.5-rad loop test but no maximum rejected fraction. Published applications instead emphasize connected, redundant networks and strict pixel masking: [Morishita (2021)](https://doi.org/10.1186/s40645-020-00402-7) requires no network gaps, [Watson et al. (2022)](https://doi.org/10.1029/2021JB022674) adds interferograms to prevent breaks, and [Morishita (2023)](https://doi.org/10.1186/s40645-023-00597-5) requires retained pixels to have no gaps, no loop errors, and no unchecked interferograms. The pre-inversion hard gate therefore tests a long, acquisition-rich largest component and requires every retained edge to participate in a triangular closure loop. Its numeric 90% acquisition, 95% period, and 2.0 edge/image limits are explicitly project-specific safeguards for the original 2020–2023 Jakarta objective, not claimed literature constants. Any inversion output remains diagnostic until uncertainty masking and independent GNSS/levelling validation are complete.

Run the notebook from a fresh Colab runtime. The wrapped-phase download is approximately 6.9 GB and re-unwrapping 609 pairs can take several hours. Checkpoints are written to Drive and verified before reuse.


In [ ]:
# 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2 — Install pinned LiCSBAS, licsar_extra, and SNAPHU
import os, sys, subprocess, pathlib

LICSBAS_REPO = pathlib.Path('/content/LiCSBAS')
LICSBAS_COMMIT = 'a145e6c4217b29eb7348ea95630454a099809098'
LICSAR_EXTRA_COMMIT = '557a50811b03a722c7a326c62ea0375949f0b7e4'

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'snaphu', 'gdal-bin', 'libgdal-dev', 'python3-gdal'], check=True)
if not LICSBAS_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/comet-licsar/LiCSBAS.git', str(LICSBAS_REPO)], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'checkout', '--detach', LICSBAS_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', str(LICSBAS_REPO), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == LICSBAS_COMMIT, (actual_commit, LICSBAS_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'h5py', 'statsmodels', 'astropy', 'cmcrameri', 'bottleneck', 'rasterio', 'requests',
    'beautifulsoup4', 'psutil', 'netCDF4', 'colorcet', 'xarray', 'rioxarray', 'pandas',
    'scipy', 'scikit-learn', 'affine',
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
    f'git+https://github.com/comet-licsar/licsar_extra.git@{LICSAR_EXTRA_COMMIT}'
], check=True)
try:
    from osgeo import gdal
except ImportError:
    gdal_version = subprocess.check_output(['gdal-config', '--version'], text=True).strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'GDAL=={gdal_version}'], check=True)
    from osgeo import gdal
print('GDAL:', gdal.VersionInfo())
if shutil_path := subprocess.run(['which', 'snaphu'], text=True, stdout=subprocess.PIPE).stdout.strip():
    print('SNAPHU:', shutil_path)
else:
    raise RuntimeError('SNAPHU installation did not produce an executable.')
print('LiCSBAS commit:', actual_commit)
print('licsar_extra commit:', LICSAR_EXTRA_COMMIT)


In [ ]:
# 3 — Fixed experiment configuration and helpers
import json, hashlib, datetime, re, shutil, math, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import requests

FRAME = '098A_09673_121312'
TRACK = '98'
START = '20200101'
END = '20231231'
DATE_TAG = f'{START}_{END}'
NLOOK = 5
ROI_BOUNDS = (106.45, -6.54, 107.22, -6.01)  # west, south, east, north
ROI_G_OPTION = f'{ROI_BOUNDS[0]}/{ROI_BOUNDS[2]}/{ROI_BOUNDS[1]}/{ROI_BOUNDS[3]}'
EXPECTED_COMPLETE_PAIRS = 609
EXPECTED_ACQUISITIONS = 120
WRAPPED_PRODUCT = 'diff_unfiltered_pha'
WRAPPED_DOWNLOAD_WORKERS = 8
WRAPPED_DOWNLOAD_ATTEMPTS = 6
WRAPPED_DOWNLOAD_BACKOFF_SECONDS = 5
UNWRAP_WORKERS = 2
UNWRAP_FILTER = 'gold'
UNWRAP_CONSISTENCY_THRESHOLD = 0.3
UNWRAP_CASCADE = 'single_10x_layer'
USE_GACOS = False
LOOP_MODE = 'multi_prime'
LOOP_THRESHOLD_RAD = 1.5

# Project-specific coverage/redundancy gates. The 25% value is retained only as
# a historical comparison because LiCSBAS literature does not define a universal
# maximum rejected-pair percentage.
HISTORICAL_MAX_REJECTION_FRACTION = 0.25
MIN_LCC_IMAGE_FRACTION = 0.90
MIN_LCC_TEMPORAL_SPAN_FRACTION = 0.95
MIN_LCC_EDGE_TO_IMAGE_RATIO = 2.0
MAX_LCC_BRIDGE_INTERFEROGRAMS = 0
MAX_LCC_INTERFEROGRAMS_WITHOUT_TRIANGLE = 0

WORK = Path('/content/insar_reunw_v9')
DRIVE_OUT = Path('/content/drive/MyDrive/InSAR_Risk_Zoning_Project')
ARCHIVE = DRIVE_OUT / 'LiCSBAS_archive_2020_2023'
GEOC_NAME = 'GEOC'
GEOCML_NAME = f'GEOCml{NLOOK}clip'
TS_NAME = f'TS_{GEOCML_NAME}_reunw_mp'
RUN_TAG = f'reunw_unfiltered_roi_ml{NLOOK}_mp'
RUN_ROLE = 'consistent_full_stack_reunwrapping_fixed_jakarta_roi'
BIN = LICSBAS_REPO / 'bin'
LIB = LICSBAS_REPO / 'LiCSBAS_lib'
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE.mkdir(parents=True, exist_ok=True)

GEOC_ARCHIVE = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.tar.gz'
GEOC_ARCHIVE_SHA = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.sha256'
PHASE_INVENTORY = ARCHIVE / f'CEDA_{FRAME}_{DATE_TAG}_{WRAPPED_PRODUCT}_inventory.json'
GEOCML_ARCHIVE = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_checkpoint.tar.gz'
GEOCML_ARCHIVE_SHA = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_checkpoint.sha256'
GEOCML_INVENTORY = ARCHIVE / f'{GEOCML_NAME}_{RUN_TAG}_{DATE_TAG}_inventory.json'
QC_SUMMARY = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_qc_summary.json'
CEDA_DATA = f'https://dap.ceda.ac.uk/neodc/comet/data/licsar_products/{TRACK}/{FRAME}'

run_env = os.environ.copy()
run_env['PYTHONPATH'] = os.pathsep.join([str(LIB), run_env.get('PYTHONPATH', '')]).rstrip(os.pathsep)

def resolve_script(role, preferred, fragments):
    exact = BIN / preferred
    if exact.exists():
        return exact
    candidates = sorted(p for p in BIN.glob('LiCSBAS*.py') if all(x.lower() in p.name.lower() for x in fragments))
    if len(candidates) != 1:
        raise RuntimeError(f'Cannot resolve {role}: {candidates}')
    return candidates[0]

SCRIPTS = {
    'reunwrap': resolve_script('reunwrap', 'LiCSBAS02to05_unwrap.py', ('02to05', 'unwrap')),
    'check': resolve_script('check', 'LiCSBAS11_check_unw.py', ('11', 'check_unw')),
    'loop': resolve_script('loop', 'LiCSBAS12_loop_closure.py', ('12', 'loop')),
    'invert': resolve_script('invert', 'LiCSBAS13_sb_inv.py', ('13', 'sb_inv')),
    'vstd': resolve_script('vstd', 'LiCSBAS14_vel_std.py', ('14', 'vel_std')),
    'geotiff': resolve_script('GeoTIFF export', 'LiCSBAS_flt2geotiff.py', ('flt2geotiff',)),
}

def run_script(role, args, required=()):
    cmd = [sys.executable, str(SCRIPTS[role]), *map(str, args)]
    print('RUN:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=WORK, env=run_env)
    if completed.returncode != 0:
        raise RuntimeError(f'{role} failed with return code {completed.returncode}')
    missing = [str(p) for p in required if not Path(p).exists()]
    if missing:
        raise RuntimeError(f'{role} completed but required output is missing: {missing}')
    return completed

PAIR_RE = re.compile(r'^\d{8}_\d{8}$')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def pair_dirs(root):
    root = Path(root)
    return sorted(p for p in root.iterdir() if p.is_dir() and PAIR_RE.fullmatch(p.name)) if root.exists() else []

def regenerated_inventory(root):
    pairs = pair_dirs(root)
    missing_unw = [p.name for p in pairs if not (p / f'{p.name}.unw').exists()]
    missing_cc = [p.name for p in pairs if not (p / f'{p.name}.cc').exists()]
    missing_conncomp = [p.name for p in pairs if not (p / f'{p.name}.conncomp').exists()]
    return {
        'root': str(root), 'pair_folders': len(pairs),
        'complete_pairs': len(pairs) - len(set(missing_unw + missing_cc + missing_conncomp)),
        'missing_unw': missing_unw, 'missing_cc': missing_cc,
        'missing_conncomp': missing_conncomp,
    }

print(json.dumps({k: str(v) for k, v in SCRIPTS.items()}, indent=2))


In [ ]:
# 4 — Validate official commands and the fixed method
for role, script in SCRIPTS.items():
    help_run = subprocess.run(
        [sys.executable, str(script), '-h'], cwd=WORK, env=run_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if help_run.returncode not in (0, 1):
        raise RuntimeError(f'{role} help check failed ({help_run.returncode})')
    print(f'--- {role}: {script.name} ---')
    print('\n'.join(help_run.stdout.splitlines()[:14]))

unwrap_help = subprocess.run(
    [sys.executable, str(SCRIPTS['reunwrap']), '-h'], cwd=WORK, env=run_env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
).stdout
for required in ('--filter', '--cascade', '--thres', '--n_para', '--gacos'):
    if required not in unwrap_help:
        raise RuntimeError(f'Official re-unwrapper does not advertise required option {required}.')
if USE_GACOS:
    raise RuntimeError('This registered experiment must not enable partial GACOS coverage.')
print('Registered method: all-pair unfiltered phase, Goldstein, single cascade, no GACOS, ROI factor 5.')


## Recovery and source-data preflight

The original verified archive supplies coherence, land mask, geometry, and metadata. Its historical unwrapped phase is retained for audit but is not read by the official re-unwrapping command. The next cells verify the archive, inventory the exact 609-pair network, confirm every unfiltered wrapped phase at CEDA, and download only missing wrapped files with atomic completion checks.


In [ ]:
# 5 — Restore the checksum-verified source archive for CC and metadata
if not GEOC_ARCHIVE.exists() or not GEOC_ARCHIVE_SHA.exists():
    raise FileNotFoundError('The verified 6.40 GB GEOC archive and checksum are required in Drive.')
expected_archive_sha = GEOC_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
archive_sha = sha256_file(GEOC_ARCHIVE)
if archive_sha != expected_archive_sha:
    raise RuntimeError('The source GEOC archive checksum does not match; refusing to restore it.')
if not (WORK / GEOC_NAME).exists():
    subprocess.run(['tar', '-xzf', str(GEOC_ARCHIVE), '-C', str(WORK)], check=True)

geoc_root = WORK / GEOC_NAME
source_pairs = pair_dirs(geoc_root)
if len(source_pairs) != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Restored GEOC contains {len(source_pairs)} pair folders; expected {EXPECTED_COMPLETE_PAIRS}.')
missing_cc = [p.name for p in source_pairs if not (p / f'{p.name}.geo.cc.tif').exists()]
if missing_cc:
    raise RuntimeError(f'Restored GEOC is missing {len(missing_cc)} coherence rasters.')
all_ifgs_expected = [p.name for p in source_pairs]
all_images_expected = sorted({d for pair in all_ifgs_expected for d in pair.split('_')})
if len(all_images_expected) != EXPECTED_ACQUISITIONS:
    raise RuntimeError(f'Expected {EXPECTED_ACQUISITIONS} acquisitions, found {len(all_images_expected)}.')
print(f'VERIFIED source archive: {len(source_pairs)} pairs, {len(all_images_expected)} acquisitions')
print('SHA-256:', archive_sha)


In [ ]:
# 6 — Audit all wrapped phases and GACOS epochs at CEDA
session = requests.Session()
session.headers.update({'User-Agent': 'Jakarta-InSAR-v9-readonly-preflight'})

def head_url(item):
    key, url = item
    try:
        response = session.head(url, allow_redirects=True, timeout=45)
        return key, url, response.status_code, int(response.headers.get('Content-Length') or 0), response.headers.get('Last-Modified')
    except Exception as error:
        return key, url, None, 0, f'{type(error).__name__}: {error}'

phase_requests = []
for pair in all_ifgs_expected:
    name = f'{pair}.geo.{WRAPPED_PRODUCT}.tif'
    phase_requests.append((pair, f'{CEDA_DATA}/{pair}/{name}'))
gacos_requests = [(date, f'{CEDA_DATA}/epochs/{date}/{date}.sltd.geo.tif') for date in all_images_expected]

def audit_requests(items, workers=16):
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(head_url, item) for item in items]
        for future in as_completed(futures):
            results.append(future.result())
    return sorted(results)

phase_audit = audit_requests(phase_requests)
gacos_audit = audit_requests(gacos_requests)
phase_available = [item for item in phase_audit if item[2] == 200 and item[3] > 0]
phase_missing = [item for item in phase_audit if item[2] != 200 or item[3] <= 0]
gacos_available = [item for item in gacos_audit if item[2] == 200 and item[3] > 0]
gacos_missing = [item for item in gacos_audit if item[2] != 200 or item[3] <= 0]
if phase_missing:
    raise RuntimeError(f'CEDA lacks {len(phase_missing)} required {WRAPPED_PRODUCT} files: {[x[0] for x in phase_missing[:10]]}')
if len(phase_available) != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Wrapped-phase audit found {len(phase_available)} files; expected {EXPECTED_COMPLETE_PAIRS}.')

phase_inventory = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'start': START, 'end': END,
    'ceda_dataset': 'LiCSAR interferometry products',
    'ceda_base_url': CEDA_DATA,
    'wrapped_product': WRAPPED_PRODUCT,
    'wrapped_pairs_available': len(phase_available),
    'wrapped_total_bytes': sum(x[3] for x in phase_available),
    'gacos_epochs_available': len(gacos_available),
    'gacos_epochs_missing': [x[0] for x in gacos_missing],
    'gacos_used': USE_GACOS,
    'phase_files': [
        {'pair': key, 'url': url, 'bytes': size, 'last_modified': modified}
        for key, url, status, size, modified in phase_available
    ],
}
PHASE_INVENTORY.write_text(json.dumps(phase_inventory, indent=2), encoding='utf-8')
print(f'Wrapped phase: {len(phase_available)}/{EXPECTED_COMPLETE_PAIRS}, {phase_inventory["wrapped_total_bytes"] / 1e9:.2f} GB')
print(f'GACOS: {len(gacos_available)}/{EXPECTED_ACQUISITIONS}; not used because coverage is incomplete')
print('Phase inventory:', PHASE_INVENTORY)


In [ ]:
# 7 — Download the identical unfiltered wrapped-phase product for all 609 pairs
phase_size_by_pair = {item['pair']: item['bytes'] for item in phase_inventory['phase_files']}

def download_phase(pair):
    destination = geoc_root / pair / f'{pair}.geo.{WRAPPED_PRODUCT}.tif'
    expected_size = phase_size_by_pair[pair]
    if destination.exists() and destination.stat().st_size == expected_size:
        return pair, 'existing', expected_size
    partial = destination.with_suffix(destination.suffix + '.partial')
    url = f'{CEDA_DATA}/{pair}/{destination.name}'
    last_error = None
    for attempt in range(1, WRAPPED_DOWNLOAD_ATTEMPTS + 1):
        if partial.exists():
            partial.unlink()
        try:
            with requests.get(url, stream=True, timeout=(45, 300), headers={'User-Agent': 'Jakarta-InSAR-v9-downloader'}) as response:
                response.raise_for_status()
                with open(partial, 'wb') as stream:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            stream.write(chunk)
            actual_size = partial.stat().st_size
            if actual_size != expected_size:
                raise RuntimeError(f'Incomplete wrapped phase for {pair}: {actual_size} != {expected_size}')
            os.replace(partial, destination)
            return pair, 'downloaded', expected_size
        except (requests.RequestException, OSError, RuntimeError) as exc:
            last_error = exc
            if partial.exists():
                partial.unlink()
            if attempt == WRAPPED_DOWNLOAD_ATTEMPTS:
                break
            delay = min(60, WRAPPED_DOWNLOAD_BACKOFF_SECONDS * (2 ** (attempt - 1)))
            print(f'Retrying {pair} after attempt {attempt}/{WRAPPED_DOWNLOAD_ATTEMPTS} failed: {type(exc).__name__}; waiting {delay}s', flush=True)
            time.sleep(delay)
    raise RuntimeError(f'Wrapped-phase download failed for {pair} after {WRAPPED_DOWNLOAD_ATTEMPTS} attempts') from last_error

download_results = []
with ThreadPoolExecutor(max_workers=WRAPPED_DOWNLOAD_WORKERS) as pool:
    futures = [pool.submit(download_phase, pair) for pair in all_ifgs_expected]
    for index, future in enumerate(as_completed(futures), 1):
        result = future.result()
        download_results.append(result)
        if index % 25 == 0 or index == EXPECTED_COMPLETE_PAIRS:
            print(f'Wrapped phase ready: {index}/{EXPECTED_COMPLETE_PAIRS}', flush=True)

phase_paths = [geoc_root / pair / f'{pair}.geo.{WRAPPED_PRODUCT}.tif' for pair in all_ifgs_expected]
bad_sizes = [p.name for p in phase_paths if not p.exists() or p.stat().st_size != phase_size_by_pair[p.parent.name]]
if bad_sizes:
    raise RuntimeError(f'Wrapped-phase size validation failed: {bad_sizes[:10]}')
print('All 609 unfiltered wrapped phases are present and size-verified.')


In [ ]:
# 8 — Regenerate all 609 ROI interferograms with the official LiCSBAS route
geocml_root = WORK / GEOCML_NAME
GEOCML_RESTORED = False
if not geocml_root.exists() and GEOCML_ARCHIVE.exists() and GEOCML_ARCHIVE_SHA.exists():
    expected_sha = GEOCML_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_sha = sha256_file(GEOCML_ARCHIVE)
    if actual_sha != expected_sha:
        raise RuntimeError('The regenerated checkpoint checksum does not match; refusing to restore it.')
    subprocess.run(['tar', '-xzf', str(GEOCML_ARCHIVE), '-C', str(WORK)], check=True)
    GEOCML_RESTORED = True
    print('Restored regenerated checkpoint:', GEOCML_ARCHIVE)

existing_regenerated = regenerated_inventory(geocml_root)
if not geocml_root.exists() or existing_regenerated['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    if geocml_root.exists():
        print(f'Resuming incomplete regenerated stack: {existing_regenerated["complete_pairs"]}/{EXPECTED_COMPLETE_PAIRS} pairs complete.')
    run_script('reunwrap', [
        '-i', WORK, '-M', NLOOK, '-g', ROI_G_OPTION,
        '--filter', UNWRAP_FILTER, '--cascade',
        '--thres', UNWRAP_CONSISTENCY_THRESHOLD,
        '--n_para', UNWRAP_WORKERS,
    ], required=[geocml_root])
else:
    print(f'{GEOCML_NAME} is complete; validating instead of repeating re-unwrapping.')

geocml_inventory = regenerated_inventory(geocml_root)
print(json.dumps(geocml_inventory, indent=2))
if geocml_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Regenerated stack has {geocml_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')

dem_values = {}
for line in (geocml_root / 'EQA.dem_par').read_text(encoding='utf-8', errors='replace').splitlines():
    fields = line.replace(':', ' ').split()
    if len(fields) >= 2 and fields[0] in {'width', 'nlines', 'post_lat', 'post_lon', 'corner_lat', 'corner_lon'}:
        dem_values[fields[0]] = fields[1]
resolution_summary = {
    'requested_roi_bounds_wsen': list(ROI_BOUNDS),
    'multilook_factor': NLOOK,
    'multilooked_parameters': dem_values,
    'nominal_effective_resolution_metres': 553,
}
reunwrap_provenance = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'start': START, 'end': END,
    'run_role': RUN_ROLE, 'source_archive_sha256': archive_sha,
    'wrapped_product': WRAPPED_PRODUCT,
    'wrapped_pairs': EXPECTED_COMPLETE_PAIRS,
    'wrapped_phase_inventory': str(PHASE_INVENTORY),
    'licsbas_commit': actual_commit,
    'licsar_extra_commit': LICSAR_EXTRA_COMMIT,
    'unwrapper': SCRIPTS['reunwrap'].name,
    'snaphu': True, 'filter': UNWRAP_FILTER,
    'cascade': UNWRAP_CASCADE,
    'consistency_threshold': UNWRAP_CONSISTENCY_THRESHOLD,
    'gacos_used': USE_GACOS,
    'gacos_available_acquisitions_at_preflight': phase_inventory['gacos_epochs_available'],
    'roi_bounds_wsen': list(ROI_BOUNDS), 'multilook': NLOOK,
    'loop_mode': LOOP_MODE, 'loop_threshold_rad': LOOP_THRESHOLD_RAD,
    'historical_source_unwrapped_phase_used': False,
}
(geocml_root / 'reunwrap_provenance.json').write_text(json.dumps(reunwrap_provenance, indent=2), encoding='utf-8')
geocml_inventory.update({'provenance': reunwrap_provenance, 'resolution': resolution_summary})
GEOCML_INVENTORY.write_text(json.dumps(geocml_inventory, indent=2), encoding='utf-8')

if not GEOCML_RESTORED:
    partial = GEOCML_ARCHIVE.with_suffix(GEOCML_ARCHIVE.suffix + '.partial')
    run = subprocess.run(['tar', '-czf', str(partial), '-C', str(WORK), GEOCML_NAME])
    if run.returncode != 0 or not partial.exists() or partial.stat().st_size == 0:
        raise RuntimeError('Regenerated checkpoint creation failed.')
    os.replace(partial, GEOCML_ARCHIVE)
checkpoint_sha = sha256_file(GEOCML_ARCHIVE)
GEOCML_ARCHIVE_SHA.write_text(f'{checkpoint_sha}  {GEOCML_ARCHIVE.name}\n', encoding='utf-8')
subprocess.run(['tar', '-tzf', str(GEOCML_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
print(f'VERIFIED regenerated checkpoint: {GEOCML_ARCHIVE} ({GEOCML_ARCHIVE.stat().st_size / 1e9:.2f} GB)')
print('SHA-256:', checkpoint_sha)


In [ ]:
# 9 — Standard quality check, 1.5-rad multi-prime loop closure, and redundancy gate
run_script('check', ['-d', GEOCML_NAME, '-t', TS_NAME], required=[WORK / TS_NAME])
run_script('loop', [
    '-d', GEOCML_NAME, '-t', TS_NAME,
    '-l', str(LOOP_THRESHOLD_RAD), '--multi_prime',
], required=[WORK / TS_NAME / 'info' / '12bad_ifg.txt'])

def data_lines(path):
    path = Path(path)
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip() and not line.lstrip().startswith('#')]

def interferogram_ids(lines, label):
    parsed, unparsed = [], []
    for line in lines:
        match = re.search(r'(?<!\d)(\d{8}_\d{8})(?!\d)', line)
        (parsed if match else unparsed).append(match.group(1) if match else line)
    if unparsed:
        raise RuntimeError(f'Cannot parse interferogram IDs from {label}: {unparsed[:5]}')
    return set(parsed)

info_dir = WORK / TS_NAME / 'info'
bad11 = interferogram_ids(data_lines(info_dir / '11bad_ifg.txt'), '11bad_ifg.txt')
bad12 = interferogram_ids(data_lines(info_dir / '12bad_ifg.txt'), '12bad_ifg.txt')
all_ifgs = sorted(p.name for p in pair_dirs(geocml_root))
if len(all_ifgs) != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Network gate found {len(all_ifgs)} interferograms; expected {EXPECTED_COMPLETE_PAIRS}.')
all_ifg_set = set(all_ifgs)
rejected_ifgs = bad11 | bad12
if rejected_ifgs - all_ifg_set:
    raise RuntimeError('Bad-interferogram files reference unknown pairs.')
retained_ifgs = sorted(all_ifg_set - rejected_ifgs)
all_images = sorted({d for pair in all_ifgs for d in pair.split('_')})

adjacency = {d: set() for d in all_images}
for pair in retained_ifgs:
    a, b = pair.split('_')
    adjacency[a].add(b); adjacency[b].add(a)
components, unseen = [], set(all_images)
while unseen:
    start = min(unseen); stack = [start]; component = set()
    while stack:
        node = stack.pop()
        if node in component: continue
        component.add(node); unseen.discard(node)
        stack.extend(adjacency[node] - component)
    components.append(sorted(component))
components.sort(key=lambda c: (-len(c), c[0]))
lcc = components[0]
lcc_set = set(lcc)
lcc_ifgs = sorted(pair for pair in retained_ifgs if set(pair.split('_')).issubset(lcc_set))

# Tarjan bridge detection: a bridge has no alternative cycle and therefore no
# loop redundancy. Retained production edges must all be cycle-checkable.
edge_name = {frozenset(pair.split('_')): pair for pair in lcc_ifgs}
disc, low, parent, bridges, clock = {}, {}, {}, [], [0]
def visit(node):
    disc[node] = low[node] = clock[0]; clock[0] += 1
    for neighbour in sorted(adjacency[node] & lcc_set):
        if neighbour not in disc:
            parent[neighbour] = node; visit(neighbour)
            low[node] = min(low[node], low[neighbour])
            if low[neighbour] > disc[node]:
                bridges.append(edge_name[frozenset((node, neighbour))])
        elif parent.get(node) != neighbour:
            low[node] = min(low[node], disc[neighbour])
visit(lcc[0])
lcc_ifgs_without_triangle = []
for pair in lcc_ifgs:
    first, second = pair.split('_')
    if not (adjacency[first] & adjacency[second] & lcc_set):
        lcc_ifgs_without_triangle.append(pair)

requested_days = (datetime.datetime.strptime(END, '%Y%m%d') - datetime.datetime.strptime(START, '%Y%m%d')).days
lcc_days = (datetime.datetime.strptime(lcc[-1], '%Y%m%d') - datetime.datetime.strptime(lcc[0], '%Y%m%d')).days if len(lcc) > 1 else 0
lcc_image_fraction = len(lcc) / len(all_images)
lcc_span_fraction = lcc_days / requested_days
lcc_edge_to_image_ratio = len(lcc_ifgs) / len(lcc)
lcc_cycle_rank = len(lcc_ifgs) - len(lcc) + 1
rejection_fraction = len(rejected_ifgs) / len(all_ifgs)

network_gate = {
    'gate_basis': 'literature_aligned_connected_cycle_redundant_network',
    'loop_mode': LOOP_MODE, 'loop_threshold_rad': LOOP_THRESHOLD_RAD,
    'total_interferograms': len(all_ifgs),
    'step11_bad_interferograms': len(bad11),
    'step12_bad_interferograms': len(bad12),
    'unique_rejected_interferograms': len(rejected_ifgs),
    'rejection_fraction': rejection_fraction,
    'historical_25pct_ceiling_exceeded': rejection_fraction > HISTORICAL_MAX_REJECTION_FRACTION,
    'rejection_fraction_role': 'reported_comparison_not_universal_hard_gate',
    'retained_interferograms': len(retained_ifgs),
    'total_images': len(all_images),
    'connected_components_including_isolates': len(components),
    'largest_connected_component': {
        'images': lcc, 'image_count': len(lcc),
        'image_fraction': lcc_image_fraction,
        'first_image': lcc[0], 'last_image': lcc[-1],
        'temporal_span_fraction': lcc_span_fraction,
        'interferogram_count': len(lcc_ifgs),
        'edge_to_image_ratio': lcc_edge_to_image_ratio,
        'cycle_rank': lcc_cycle_rank,
        'bridge_interferograms': sorted(bridges),
        'interferograms_without_triangle_loop': sorted(lcc_ifgs_without_triangle),
    },
    'thresholds': {
        'min_lcc_image_fraction': MIN_LCC_IMAGE_FRACTION,
        'min_lcc_temporal_span_fraction': MIN_LCC_TEMPORAL_SPAN_FRACTION,
        'min_lcc_edge_to_image_ratio': MIN_LCC_EDGE_TO_IMAGE_RATIO,
        'max_lcc_bridge_interferograms': MAX_LCC_BRIDGE_INTERFEROGRAMS,
        'max_lcc_interferograms_without_triangle': MAX_LCC_INTERFEROGRAMS_WITHOUT_TRIANGLE,
    },
    'post_inversion_requirements': ['n_gap=0 for promoted pixels', 'n_loop_err=0 for promoted pixels', 'uncertainty sensitivity', 'independent GNSS_or_levelling_validation'],
}
gate_failures = []
if lcc_image_fraction < MIN_LCC_IMAGE_FRACTION:
    gate_failures.append(f'LCC retains only {lcc_image_fraction:.1%} of acquisitions')
if lcc_span_fraction < MIN_LCC_TEMPORAL_SPAN_FRACTION:
    gate_failures.append(f'LCC spans only {lcc_span_fraction:.1%} of requested period')
if lcc_edge_to_image_ratio < MIN_LCC_EDGE_TO_IMAGE_RATIO:
    gate_failures.append(f'LCC edge/image ratio is only {lcc_edge_to_image_ratio:.2f}')
if len(bridges) > MAX_LCC_BRIDGE_INTERFEROGRAMS:
    gate_failures.append(f'LCC contains {len(bridges)} bridge interferograms without cycle redundancy')
if len(lcc_ifgs_without_triangle) > MAX_LCC_INTERFEROGRAMS_WITHOUT_TRIANGLE:
    gate_failures.append(f'LCC contains {len(lcc_ifgs_without_triangle)} interferograms without a triangular closure loop')
network_gate['passed'] = not gate_failures
network_gate['gate_failures'] = gate_failures

NETWORK_GATE_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_network_gate.json'
NETWORK_GATE_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_network_gate.json').write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
(info_dir / f'{RUN_TAG}_reunwrap_provenance.json').write_text(json.dumps(reunwrap_provenance, indent=2), encoding='utf-8')
print(json.dumps(network_gate, indent=2))

DIAGNOSTIC_ARCHIVE = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.tar.gz'
DIAGNOSTIC_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.sha256'
diagnostic_candidates = [f'{TS_NAME}/info', f'{TS_NAME}/network', f'{TS_NAME}/12loop', f'{GEOCML_NAME}/reunwrap_provenance.json']
diagnostic_items = [item for item in diagnostic_candidates if (WORK / item).exists()]
partial = DIAGNOSTIC_ARCHIVE.with_suffix(DIAGNOSTIC_ARCHIVE.suffix + '.partial')
run = subprocess.run(['tar', '-czf', str(partial), '-C', str(WORK), *diagnostic_items])
if run.returncode != 0 or not partial.exists() or partial.stat().st_size == 0:
    raise RuntimeError('Step-12 diagnostic archive failed.')
os.replace(partial, DIAGNOSTIC_ARCHIVE)
diagnostic_sha = sha256_file(DIAGNOSTIC_ARCHIVE)
DIAGNOSTIC_ARCHIVE_SHA.write_text(f'{diagnostic_sha}  {DIAGNOSTIC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED diagnostics: {DIAGNOSTIC_ARCHIVE} ({DIAGNOSTIC_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Diagnostic SHA-256:', diagnostic_sha)
if gate_failures:
    raise RuntimeError('REGENERATED NETWORK GATE FAILED: ' + '; '.join(gate_failures) + '. Diagnostics are archived; do not continue.')
print('REGENERATED NETWORK GATE PASSED — inversion may run, but outputs remain diagnostic pending validation.')


In [ ]:
# 10 — Steps 13 and 14: SBAS inversion and velocity uncertainty
run_script(
    'invert', ['-d', GEOCML_NAME, '-t', TS_NAME, '--mem_size', '4000'],
    required=[WORK / TS_NAME / 'cum.h5', WORK / TS_NAME / 'results' / 'vel'],
)
run_script(
    'vstd', ['-t', TS_NAME],
    required=[WORK / TS_NAME / 'results' / 'vstd'],
)


In [ ]:
# 11 — Export the matched four-layer GeoTIFF stack with exact dates and factor label
results_dir = WORK / TS_NAME / 'results'
dem_par = WORK / GEOCML_NAME / 'EQA.dem_par'
layers = ('vel', 'vstd', 'coh_avg', 'n_unw')
exported = {}
for layer in layers:
    source = results_dir / layer
    if not source.exists():
        raise FileNotFoundError(f'Missing LiCSBAS result layer: {source}')
    destination = DRIVE_OUT / f'jakarta_{layer}_{DATE_TAG}_{RUN_TAG}.tif'
    run_script(
        'geotiff', ['-i', source, '-p', dem_par, '-o', destination],
        required=[destination],
    )
    exported[layer] = destination
print(json.dumps({k: str(v) for k, v in exported.items()}, indent=2))


In [ ]:
# 12 — Validate output grids and write diagnostic provenance
import numpy as np
import rasterio

layer_meta, grid_key = {}, None
for role, path in exported.items():
    with rasterio.open(path) as src:
        data = src.read(1)
        current_grid = (src.width, src.height, str(src.crs), tuple(src.transform))
        if grid_key is None:
            grid_key = current_grid
        elif current_grid != grid_key:
            raise RuntimeError(f'Grid mismatch for {role}: {current_grid} != {grid_key}')
        finite = np.isfinite(data)
        values = data[finite]
        if role == 'vstd' and (not finite.any() or np.any(values < 0)):
            raise RuntimeError('vstd is empty or contains negative values')
        layer_meta[role] = {
            'path': str(path), 'sha256': sha256_file(path), 'bytes': path.stat().st_size,
            'shape': [src.height, src.width], 'crs': str(src.crs),
            'transform': list(src.transform), 'resolution': list(src.res),
            'total_pixels': int(data.size), 'finite_pixels': int(finite.sum()),
            'finite_fraction': float(finite.mean()),
            'min': float(values.min()) if values.size else None,
            'median': float(np.median(values)) if values.size else None,
            'max': float(values.max()) if values.size else None,
        }

with rasterio.open(exported['vel']) as vel_src, rasterio.open(exported['vstd']) as vstd_src:
    common_valid = np.isfinite(vel_src.read(1)) & np.isfinite(vstd_src.read(1))

quality_flags = []
if layer_meta['vel']['finite_fraction'] < 0.50:
    quality_flags.append('velocity coverage is below 50% of the output grid')
if layer_meta['coh_avg']['median'] is not None and layer_meta['coh_avg']['median'] < 0.20:
    quality_flags.append('median average coherence is below 0.20')
if layer_meta['vstd']['median'] is not None and layer_meta['vstd']['median'] >= 12.0:
    quality_flags.append('median vstd is close to or above the planned 15 mm/year threshold')
if layer_meta['vel']['min'] is not None and max(abs(layer_meta['vel']['min']), abs(layer_meta['vel']['max'])) > 200.0:
    quality_flags.append('absolute velocity extremes exceed 200 mm/year and require validation')

qc = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'scientific_status': 'diagnostic_regenerated_candidate_pending_pixel_qc_and_external_validation',
    'run_role': RUN_ROLE, 'frame': FRAME, 'orbit': 'ascending',
    'start': START, 'end': END, 'multilook': NLOOK, 'run_tag': RUN_TAG,
    'reunwrap_provenance': reunwrap_provenance,
    'network_survival_gate': network_gate,
    'common_valid_vel_vstd_pixels': int(common_valid.sum()),
    'common_valid_vel_vstd_fraction': float(common_valid.mean()),
    'resolution': resolution_summary, 'layer_statistics': layer_meta,
    'quality_flags': quality_flags,
    'promotion_requirements': [
        'mask final pixels to n_gap=0 and n_loop_err=0',
        'complete uncertainty/coherence sensitivity',
        'validate against date-compatible GNSS or levelling',
        'derive explicit exposure before calling the combined output risk',
    ],
}
QC_SUMMARY.write_text(json.dumps(qc, indent=2, allow_nan=False), encoding='utf-8')
manifest = {
    'created_utc': qc['created_utc'], 'frame': FRAME, 'orbit': 'ascending',
    'start': START, 'end': END, 'multilook': NLOOK, 'run_tag': RUN_TAG,
    'run_role': RUN_ROLE, 'licsbas_commit': actual_commit,
    'licsar_extra_commit': LICSAR_EXTRA_COMMIT,
    'source_archive': {'path': str(GEOC_ARCHIVE), 'sha256': archive_sha},
    'wrapped_phase_inventory': str(PHASE_INVENTORY),
    'regenerated_checkpoint': {'path': str(GEOCML_ARCHIVE), 'sha256': checkpoint_sha, 'complete_pairs': geocml_inventory['complete_pairs']},
    'network_survival_gate': network_gate,
    'steps': ['restore_verified_CC_and_metadata', 'download_all_unfiltered_wrapped_phase', 'LiCSBAS02to05_full_reunwrap_ROI_factor5', '11', '12', '13', '14'],
    'layers': layer_meta, 'qc_summary': str(QC_SUMMARY),
}
manifest_path = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps(qc, indent=2))
print('QC summary:', QC_SUMMARY)
print('Manifest:', manifest_path)
print('Outputs remain diagnostic until the listed promotion requirements pass.')


In [ ]:
# 13 — Archive final diagnostic results and verify them
RESULTS_ARCHIVE = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.tar.gz'
RESULTS_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.sha256'
items = [f'{TS_NAME}/results', f'{TS_NAME}/info', f'{TS_NAME}/network', f'{GEOCML_NAME}/EQA.dem_par', f'{GEOCML_NAME}/reunwrap_provenance.json']
existing_items = [item for item in items if (WORK / item).exists()]
partial = RESULTS_ARCHIVE.with_suffix(RESULTS_ARCHIVE.suffix + '.partial')
run = subprocess.run(['tar', '-czf', str(partial), '-C', str(WORK), *existing_items])
if run.returncode != 0 or not partial.exists() or partial.stat().st_size == 0:
    raise RuntimeError('Final-results archive failed')
os.replace(partial, RESULTS_ARCHIVE)
subprocess.run(['tar', '-tzf', str(RESULTS_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
results_sha = sha256_file(RESULTS_ARCHIVE)
RESULTS_ARCHIVE_SHA.write_text(f'{results_sha}  {RESULTS_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED final diagnostic archive: {RESULTS_ARCHIVE} ({RESULTS_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Final archive SHA-256:', results_sha)
print(f'DONE: exact-date {RUN_TAG} diagnostic GeoTIFFs, manifest, QC summary, and verified archives are in Drive.')
